# MDR Merchant Impact Analysis

**Portfolio-ready Python analysis**

This notebook analyzes Merchant Discount Rate (MDR) exposure across payment transactions, with emphasis on transaction eligibility, small-merchant exemptions, the ₹300 MDR cap, profitability impact, merchant concentration, transaction size, and city-level patterns.

### Final project KPIs
- **1,000** total transactions
- **308** potentially MDR-eligible transactions before the small-merchant exemption
- **55** eligible transactions exempted under the small-merchant rule
- **253** final MDR-applicable transactions
- **₹22,175.56** total MDR cost

> The notebook preserves the original analysis logic while removing exploratory/debugging cells and personal scratch notes. The original file should be kept separately as the working-analysis version.


## 1. Setup and data loading

The notebook expects the project data under `data/raw/` at the repository root. If the notebook is stored inside a `python/` folder, the first path below will resolve correctly.


In [1]:
from pathlib import Path
import pandas as pd

# Portable project path: works when the notebook is inside /python or at repo root.
DATA_DIR = Path("../data/raw")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/raw")

required_files = [
    "fact_transactions_raw.csv",
    "dim_date.csv",
    "dim_product.csv",
    "dim_customer.csv",
    "dim_merchant.csv",
    "dim_city.csv",
    "dim_payment.csv",
    "dim_mdr_rules.csv",
]

missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing project files in data/raw: " + ", ".join(missing_files)
    )

fact = pd.read_csv(DATA_DIR / "fact_transactions_raw.csv")
dates = pd.read_csv(DATA_DIR / "dim_date.csv")
products = pd.read_csv(DATA_DIR / "dim_product.csv")
customers = pd.read_csv(DATA_DIR / "dim_customer.csv")
merchants = pd.read_csv(DATA_DIR / "dim_merchant.csv")
cities = pd.read_csv(DATA_DIR / "dim_city.csv")
payments = pd.read_csv(DATA_DIR / "dim_payment.csv")
mdr_rules = pd.read_csv(DATA_DIR / "dim_mdr_rules.csv")

print("Loaded transaction and dimension tables successfully.")
print(f"Transactions: {fact.shape[0]:,} rows x {fact.shape[1]} columns")


Loaded transaction and dimension tables successfully.
Transactions: 1,000 rows x 17 columns


## 2. Data preparation and quality checks

The source data contains small rounding inconsistencies in the stored financial fields and whitespace inconsistencies in payment categories. The analysis standardizes those fields before applying MDR logic.


In [2]:
# Recalculate financial fields from the stored source components.
fact["Transaction_Amount"] = (
    fact["Gross_Sales_Amount"] - fact["Discount"]
).round(2)

fact["Gross_Profit"] = (
    fact["Transaction_Amount"] - fact["Cost_of_Goods"]
).round(2)

# Standardize payment categories.
fact["Payment_Method"] = fact["Payment_Method"].str.strip()
fact["UPI_Type"] = fact["UPI_Type"].str.strip()

# Standardize dates.
fact["Transaction_Date"] = pd.to_datetime(fact["Transaction_Date"])


In [3]:
# Basic quality profile.
quality_summary = pd.Series({
    "Rows": len(fact),
    "Columns": fact.shape[1],
    "Duplicate Transaction_IDs": fact["Transaction_ID"].duplicated().sum(),
    "Missing Customer_ID": fact["Customer_ID"].isna().sum(),
    "Missing Transaction_Date": fact["Transaction_Date"].isna().sum(),
})
quality_summary


Rows                         1000
Columns                        17
Duplicate Transaction_IDs       0
Missing Customer_ID              3
Missing Transaction_Date         0
dtype: int64

### Missing values and transaction status

The three missing `Customer_ID` values occur on failed transactions. Those transactions are retained for auditability, but failed transactions are excluded from completed-sales and MDR profitability calculations. `UPI_Type` is expected to be null for non-UPI payment methods.


In [4]:
status_summary = (
    fact.groupby("Transaction_Status")["Transaction_Amount"]
        .agg(["count", "sum", "mean"])
        .round(2)
)
status_summary


                    count          sum      mean
Transaction_Status                              
Failed                 39    568727.75  14582.76
Success               961  15899868.73  16545.13

### Star-schema key validation

Each foreign-key-like ID in the transaction table is checked against its corresponding dimension table before analysis.


In [5]:
relationship_checks = pd.Series({
    "Invalid Merchant IDs": (~fact["Merchant_ID"].isin(merchants["Merchant_ID"])).sum(),
    "Invalid Customer IDs": (~fact["Customer_ID"].dropna().isin(customers["Customer_ID"])).sum(),
    "Invalid Product IDs": (~fact["Product_ID"].isin(products["Product_ID"])).sum(),
    "Invalid City IDs": (~fact["City_ID"].isin(cities["City_ID"])).sum(),
    "Invalid Payment IDs": (~fact["Payment_ID"].isin(payments["Payment_ID"])).sum(),
})
relationship_checks


Invalid Merchant IDs    0
Invalid Customer IDs    0
Invalid Product IDs     0
Invalid City IDs        0
Invalid Payment IDs     0
dtype: int64

### Payment and date validation

After trimming whitespace, the dataset contains four payment methods. UPI transactions use `P2M` or `P2P`; non-UPI transactions do not carry a UPI type. The transaction period is 01 Apr 2026 through 30 Sep 2026.


In [6]:
payment_counts = fact["Payment_Method"].value_counts(dropna=False)
upi_type_counts = fact["UPI_Type"].value_counts(dropna=False)

invalid_upi = (
    (fact["Payment_Method"] == "UPI")
    & (~fact["UPI_Type"].isin(["P2M", "P2P"]))
)
invalid_non_upi = (
    (fact["Payment_Method"] != "UPI")
    & fact["UPI_Type"].notna()
)

print(payment_counts)
print("\nUPI types:")
print(upi_type_counts)
print("\nInvalid UPI combinations:", int(invalid_upi.sum()))
print("Invalid non-UPI combinations:", int(invalid_non_upi.sum()))
print("Date range:", fact["Transaction_Date"].min().date(), "to", fact["Transaction_Date"].max().date())


Payment_Method
UPI            549
Debit Card     191
Credit Card    152
Cash           108
Name: count, dtype: int64

UPI types:
UPI_Type
P2M    519
NaN    451
P2P     30
Name: count, dtype: int64

Invalid UPI combinations: 0
Invalid non-UPI combinations: 0
Date range: 2026-04-01 to 2026-09-30


## 3. MDR business rules

The modeled rules used in this project are:

1. Transaction status must be **Success**.
2. Payment method must be **UPI** and UPI type must be **P2M**.
3. Transaction amount must be **greater than ₹2,000**.
4. A qualifying small merchant is exempt when `Small_Merchant_Flag = True` and monthly UPI volume is **≤ ₹1,00,000**.
5. MDR is **0.4% of transaction value**, capped at **₹300 per transaction**.

The distinction between *potential eligibility* and *final applicability* is important: 308 transactions pass the transaction-level eligibility rules, but 55 are removed by the merchant-level exemption, leaving 253 final MDR-applicable transactions.


In [7]:
MDR_RATE = 0.004
MDR_CAP = 300.0
MIN_TRANSACTION_AMOUNT = 2000.0
SMALL_MERCHANT_LIMIT = 100000.0

successful = fact[fact["Transaction_Status"] == "Success"].copy()

successful["MDR_Eligible"] = (
    (successful["Payment_Method"] == "UPI")
    & (successful["UPI_Type"] == "P2M")
    & (successful["Transaction_Amount"] > MIN_TRANSACTION_AMOUNT)
)

print("Successful transactions:", len(successful))
print("Potential MDR-eligible transactions:", int(successful["MDR_Eligible"].sum()))


Successful transactions: 961
Potential MDR-eligible transactions: 308


### Small-merchant exemption

The merchant dimension is used to identify merchants that qualify for the modeled small-merchant exemption.


In [8]:
qualifying_small_merchants = merchants[
    (merchants["Small_Merchant_Flag"] == True)
    & (merchants["Monthly_UPI_Volume"] <= SMALL_MERCHANT_LIMIT)
].copy()

print("Qualifying small merchants:", len(qualifying_small_merchants))


Qualifying small merchants: 16


In [9]:
eligible_before_exemption = successful[
    successful["MDR_Eligible"]
].copy()

eligible_before_exemption = eligible_before_exemption.merge(
    qualifying_small_merchants[["Merchant_ID", "Monthly_UPI_Volume"]],
    on="Merchant_ID",
    how="left",
    indicator="Small_Merchant_Match",
)

eligible_before_exemption["Small_Merchant_Exempt"] = (
    eligible_before_exemption["Small_Merchant_Match"] == "both"
)

exemption_counts = eligible_before_exemption["Small_Merchant_Exempt"].value_counts()
exemption_counts


Small_Merchant_Exempt
False    253
True      55
Name: count, dtype: int64

**Observed result:** 55 potentially eligible transactions are exempt, leaving 253 final MDR-applicable transactions.


### Final MDR-applicable population

MDR is calculated only after the exemption rule is applied. The cap is applied at transaction level; aggregate totals are rounded for presentation.


In [10]:
final_mdr = eligible_before_exemption[
    ~eligible_before_exemption["Small_Merchant_Exempt"]
].copy()

final_mdr["MDR_Amount"] = (
    final_mdr["Transaction_Amount"] * MDR_RATE
).clip(upper=MDR_CAP)

final_mdr["Net_Profit_After_MDR"] = (
    final_mdr["Gross_Profit"] - final_mdr["MDR_Amount"]
)

final_summary = pd.Series({
    "Final MDR-applicable transactions": len(final_mdr),
    "Final MDR transaction value": final_mdr["Transaction_Amount"].sum(),
    "Total MDR cost": final_mdr["MDR_Amount"].sum(),
    "Gross profit before MDR": final_mdr["Gross_Profit"].sum(),
    "Net profit after MDR": final_mdr["Net_Profit_After_MDR"].sum(),
})
final_summary.round(2)


Final MDR-applicable transactions        253.00
Final MDR transaction value          6468187.23
Total MDR cost                         22175.56
Gross profit before MDR              1300597.23
Net profit after MDR                 1278421.67
dtype: float64

## 4. Profitability impact

For the 253 final MDR-applicable transactions, total MDR cost is ₹22,175.56. This reduces gross profit from approximately ₹13.01 lakh to ₹12.78 lakh, an impact of about **1.71%** on gross profit for the applicable population.


In [11]:
gross_profit = final_mdr["Gross_Profit"].sum()
total_mdr = final_mdr["MDR_Amount"].sum()
net_profit = final_mdr["Net_Profit_After_MDR"].sum()
mdr_profit_impact_pct = total_mdr / gross_profit * 100

print(f"Gross Profit Before MDR: ₹{gross_profit:,.2f}")
print(f"Total MDR Cost: ₹{total_mdr:,.2f}")
print(f"Net Profit After MDR: ₹{net_profit:,.2f}")
print(f"MDR Impact on Gross Profit: {mdr_profit_impact_pct:.2f}%")


Gross Profit Before MDR: ₹1,300,597.23
Total MDR Cost: ₹22,175.56
Net Profit After MDR: ₹1,278,421.67
MDR Impact on Gross Profit: 1.71%


## 5. Merchant-level analysis

Merchant analysis separates **absolute MDR cost** from **MDR impact relative to gross profit**. A merchant can have a moderate MDR amount but a relatively high profitability impact.


In [12]:
merchant_mdr = (
    final_mdr
    .groupby("Merchant_ID")
    .agg(
        Transactions=("Transaction_ID", "count"),
        Transaction_Value=("Transaction_Amount", "sum"),
        Gross_Profit=("Gross_Profit", "sum"),
        MDR_Cost=("MDR_Amount", "sum"),
        Net_Profit=("Net_Profit_After_MDR", "sum"),
    )
    .reset_index()
)

merchant_mdr["MDR_Impact_%"] = (
    merchant_mdr["MDR_Cost"] / merchant_mdr["Gross_Profit"] * 100
)

merchant_mdr = merchant_mdr.sort_values("MDR_Cost", ascending=False)
merchant_mdr.head(10).round(2)


   Merchant_ID  Transactions  Transaction_Value  Gross_Profit  MDR_Cost  Net_Profit  MDR_Impact_%
64        M084             7          271675.09      44065.09    861.60    43203.49          1.96
51        M067             5          332948.37      48358.37    857.01    47501.36          1.77
31        M042             4          272066.35      55296.35    816.49    54479.86          1.48
40        M056             4          248496.77      39566.77    786.18    38780.59          1.99
16        M024             6          169655.58      27385.58    678.62    26706.96          2.48
78        M100             6          198870.41      40390.41    668.22    39722.19          1.65
73        M094             4          217127.68      50347.68    613.98    49733.70          1.22
19        M028             4          136824.32      20164.32    547.30    19617.02          2.71
54        M070             6          236959.91      55159.91    546.11    54613.80          0.99
72        M092      

In [13]:
top10_mdr_cost = merchant_mdr.head(10)["MDR_Cost"].sum()
total_mdr_cost = merchant_mdr["MDR_Cost"].sum()
top10_mdr_share = top10_mdr_cost / total_mdr_cost * 100

print(f"Top 10 Merchant MDR Cost: ₹{top10_mdr_cost:,.2f}")
print(f"Total MDR Cost: ₹{total_mdr_cost:,.2f}")
print(f"Top 10 MDR Cost Share: {top10_mdr_share:.2f}%")


Top 10 Merchant MDR Cost: ₹6,904.63
Total MDR Cost: ₹22,175.56
Top 10 MDR Cost Share: 31.14%


**Business interpretation:** the top 10 merchants account for about **31.14%** of total MDR cost, so the exposure is not limited to only a small group of merchants.


## 6. Transaction-size analysis

Transaction bands help show how the ₹300 cap changes the proportional MDR burden at higher transaction values.


In [14]:
final_mdr["Transaction_Band"] = pd.cut(
    final_mdr["Transaction_Amount"],
    bins=[0, 10000, 25000, 50000, 75000, float("inf")],
    labels=["≤ ₹10K", "₹10K–₹25K", "₹25K–₹50K", "₹50K–₹75K", "> ₹75K"],
)

transaction_band_analysis = (
    final_mdr
    .groupby("Transaction_Band", observed=False)
    .agg(
        Transactions=("Transaction_ID", "count"),
        Transaction_Value=("Transaction_Amount", "sum"),
        MDR_Cost=("MDR_Amount", "sum"),
        Gross_Profit=("Gross_Profit", "sum"),
        Net_Profit=("Net_Profit_After_MDR", "sum"),
    )
    .reset_index()
)

transaction_band_analysis["MDR_Impact_%"] = (
    transaction_band_analysis["MDR_Cost"]
    / transaction_band_analysis["Gross_Profit"]
    * 100
)

transaction_band_analysis.round(2)


  Transaction_Band  Transactions  Transaction_Value  MDR_Cost  Gross_Profit  Net_Profit  MDR_Impact_%
0           ≤ ₹10K           125          581703.67   2326.81     143423.67   141096.86          1.62
1        ₹10K–₹25K            45          762109.19   3048.44     159009.19   155960.75          1.92
2        ₹25K–₹50K            53         2049013.74   8196.05     426943.74   418747.69          1.92
3        ₹50K–₹75K            10          651063.34   2604.25     114013.34   111409.09          2.28
4           > ₹75K            20         2424297.29   6000.00     457207.29   451207.29          1.31

**Observed result:** the ₹50K–₹75K band has the highest MDR impact on gross profit at about **2.28%**, while transactions above ₹75K have a lower proportional impact because the ₹300 cap limits MDR.


## 7. Payment-method analysis

All final MDR-applicable transactions in this modeled dataset are UPI P2M transactions, so payment method is not a differentiating dimension inside the final MDR population. It remains useful in the dashboard for validating that recorded MDR is concentrated in UPI.


In [15]:
payment_analysis = (
    final_mdr
    .groupby(["Payment_Method", "UPI_Type"])
    .agg(
        Transactions=("Transaction_ID", "count"),
        Transaction_Value=("Transaction_Amount", "sum"),
        MDR_Cost=("MDR_Amount", "sum"),
        Gross_Profit=("Gross_Profit", "sum"),
        Net_Profit=("Net_Profit_After_MDR", "sum"),
    )
    .reset_index()
)

payment_analysis["MDR_Impact_%"] = (
    payment_analysis["MDR_Cost"] / payment_analysis["Gross_Profit"] * 100
)

payment_analysis.round(2)


  Payment_Method UPI_Type  Transactions  Transaction_Value  MDR_Cost  Gross_Profit  Net_Profit  MDR_Impact_%
0            UPI      P2M           253         6468187.23  22175.56    1300597.23  1278421.67          1.71

## 8. City-level MDR analysis

City analysis compares absolute MDR cost with MDR impact relative to gross profit. These two measures can lead to different prioritization decisions.


In [16]:
city_mdr = (
    final_mdr
    .groupby("City_ID")
    .agg(
        Transactions=("Transaction_ID", "count"),
        Transaction_Value=("Transaction_Amount", "sum"),
        MDR_Cost=("MDR_Amount", "sum"),
        Gross_Profit=("Gross_Profit", "sum"),
        Net_Profit=("Net_Profit_After_MDR", "sum"),
    )
    .reset_index()
)

city_mdr["MDR_Impact_%"] = city_mdr["MDR_Cost"] / city_mdr["Gross_Profit"] * 100
city_mdr = city_mdr.sort_values("MDR_Cost", ascending=False)
city_mdr.head(10).round(2)


   City_ID  Transactions  Transaction_Value  MDR_Cost  Gross_Profit  Net_Profit  MDR_Impact_%
16    C017            16          737291.57   2202.61     136081.57   133878.96          1.62
10    C011            20          459320.94   1631.92      76850.94    75219.02          2.12
23    C024            12          422197.61   1572.17      93707.61    92135.44          1.68
5     C006            16          398508.93   1386.23      70698.93    69312.70          1.96
8     C009            15          348246.98   1167.88      57576.98    56409.10          2.03
18    C019            12          348007.38   1137.50      72997.38    71859.88          1.56
14    C015            10          278986.10   1106.15      59256.10    58149.95          1.87
11    C012            12          320713.09   1097.57      66323.09    65225.52          1.65
2     C003            13          271559.05   1086.24      53069.05    51982.81          2.05
15    C016            10          260604.54   1042.42      6

**Observed result:** City `C017` has the highest absolute MDR cost (about ₹2,202.61), while `C011` shows a higher MDR impact relative to gross profit. This illustrates why both absolute cost and profitability impact should be evaluated.


## 9. MDR cap analysis

At a 0.4% rate, a ₹75,000 transaction reaches the ₹300 cap. The analysis compares the actual capped MDR with the amount that would have been charged without the cap.


In [17]:
cap_analysis = final_mdr[
    final_mdr["Transaction_Amount"] >= 75000
].copy()

cap_analysis["MDR_Without_Cap"] = cap_analysis["Transaction_Amount"] * MDR_RATE
cap_analysis["MDR_Saved_By_Cap"] = (
    cap_analysis["MDR_Without_Cap"] - cap_analysis["MDR_Amount"]
)

cap_summary = pd.Series({
    "Transactions hitting cap": len(cap_analysis),
    "Transaction value": cap_analysis["Transaction_Amount"].sum(),
    "MDR without cap": cap_analysis["MDR_Without_Cap"].sum(),
    "Actual MDR": cap_analysis["MDR_Amount"].sum(),
    "MDR saved by cap": cap_analysis["MDR_Saved_By_Cap"].sum(),
})
cap_summary.round(2)


Transactions hitting cap         20.00
Transaction value          2424297.29
MDR without cap               9697.19
Actual MDR                    6000.00
MDR saved by cap              3697.19
dtype: float64

In [18]:
cap_saving_pct = (
    cap_analysis["MDR_Saved_By_Cap"].sum()
    / cap_analysis["MDR_Without_Cap"].sum()
    * 100
)

cap_share_of_total_mdr = (
    cap_analysis["MDR_Saved_By_Cap"].sum()
    / final_mdr["MDR_Amount"].sum()
    * 100
)

print(f"MDR reduction for capped transactions: {cap_saving_pct:.2f}%")
print(f"Cap savings as % of actual total MDR: {cap_share_of_total_mdr:.2f}%")


MDR reduction for capped transactions: 38.13%
Cap savings as % of actual total MDR: 16.67%


**Business interpretation:** 20 transactions reach the cap. The cap reduces their MDR by approximately **₹3,697.19**, a **38.13%** reduction versus the uncapped charge for those transactions.


## 10. Small-merchant exemption impact

The exemption analysis measures potential MDR that would have been generated if the 55 eligible transactions had not been exempted.


In [19]:
exemption_analysis = (
    eligible_before_exemption
    .assign(
        Potential_MDR=lambda df: (df["Transaction_Amount"] * MDR_RATE).clip(upper=MDR_CAP)
    )
    .groupby("Small_Merchant_Exempt")
    .agg(
        Transactions=("Transaction_ID", "count"),
        Transaction_Value=("Transaction_Amount", "sum"),
        Potential_MDR=("Potential_MDR", "sum"),
    )
    .reset_index()
)

exemption_analysis.round(2)


   Small_Merchant_Exempt  Transactions  Transaction_Value  Potential_MDR
0                  False           253         6468187.23       22175.56
1                   True            55         1029423.52        3998.84

In [20]:
exempted_mdr = exemption_analysis.loc[
    exemption_analysis["Small_Merchant_Exempt"],
    "Potential_MDR",
].iloc[0]

exemption_share = exempted_mdr / final_mdr["MDR_Amount"].sum() * 100

print(f"Potential MDR avoided through exemption: ₹{exempted_mdr:,.2f}")
print(f"Exemption amount vs actual MDR: {exemption_share:.2f}%")


Potential MDR avoided through exemption: ₹3,998.84
Exemption amount vs actual MDR: 18.03%


**Observed result:** the 55 exempted transactions represent approximately **₹3,998.84** of potential MDR. This is avoided liability under the modeled exemption rule, not an expense that was actually incurred and later recovered.


## 11. Key business insights

1. **Final MDR exposure is smaller than initial eligibility suggests.** 308 transactions are potentially eligible, but 55 small-merchant exemptions reduce the final applicable population to 253.
2. **MDR has a measurable profitability effect.** ₹22,175.56 of MDR reduces gross profit by about 1.71% across the final MDR-applicable transactions.
3. **MDR exposure is distributed across merchants.** The top 10 merchants generate about 31.14% of total MDR cost, leaving most cost outside the top 10.
4. **The ₹300 cap materially limits high-value transaction cost.** It reduces MDR by about ₹3,697.19 across 20 capped transactions.
5. **The small-merchant exemption also has a measurable effect.** The 55 exempted transactions represent about ₹3,998.84 of potential MDR avoided.
6. **Transaction size matters.** The ₹50K–₹75K band shows the highest MDR impact relative to gross profit, while transactions above ₹75K benefit from the cap.

### Portfolio interpretation
This notebook demonstrates data cleaning, relational validation, business-rule translation, grouped analysis, profitability analysis, and the separation of transaction-level eligibility from merchant-level exemption logic. The final SQL/Power BI reporting layer uses the same core business logic for dashboard validation.
